# LeafCare: train the local Sigatoka detector

Run this notebook in **Google Colab** with a GPU runtime if one is available. It trains the existing two-class detector (`Healthy`, `sigatoka`) from the project dataset and evaluates the best checkpoint on the held-out test split. Predictions are not a diagnosis.

## Before you start
1. Upload [LeafCare-combined-training.zip](./LeafCare-combined-training.zip) to Google Drive at `MyDrive/LeafCare/LeafCare-combined-training.zip` (create the `LeafCare` folder if needed).
2. Optionally upload reviewed examples as `MyDrive/LeafCare/LeafCare-reviewed-feedback.zip`. The preparation cell merges these images and YOLO labels into the training split only.
3. In Colab, choose **Runtime → Change runtime type → T4 GPU** if available, then run cells from top to bottom.

This combined dataset keeps the original validation/test splits and adds only reviewed supplementary images to training. Candidate Sigatoka boxes came from the current model, so they can still have missed lesions or errors; validation/test images from the supplementary dataset were deliberately excluded to avoid evaluating against model-generated labels. The archive includes the current checkpoint, so this run fine-tunes the current detector. The preparation cell converts remaining segmentation polygons to bounding boxes and rejects malformed labels. The original dataset export identifies its license as CC BY 4.0; retain its attribution when sharing it.

In [ ]:
%pip -q install ultralytics pyyaml

In [ ]:
from pathlib import Path
import math
import shutil
import zipfile

import yaml
from google.colab import drive

drive.mount('/content/drive')

ARCHIVE = Path('/content/drive/MyDrive/LeafCare/LeafCare-combined-training.zip')
EXTRACT_ROOT = Path('/content/leafcare_source')
CLEAN_ROOT = Path('/content/leafcare_yolo_dataset')

if not ARCHIVE.is_file():
    raise FileNotFoundError(f'Upload the combined dataset zip to: {ARCHIVE}')

EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ARCHIVE) as archive:
    root = EXTRACT_ROOT.resolve()
    for member in archive.infolist():
        target = (EXTRACT_ROOT / member.filename).resolve()
        if target != root and root not in target.parents:
            raise ValueError(f'Unsafe path in dataset archive: {member.filename}')
    archive.extractall(EXTRACT_ROOT)

yaml_candidates = [
    path for path in EXTRACT_ROOT.rglob('data.yaml')
    if all((path.parent / split).is_dir() for split in ('train', 'valid', 'test'))
]
if len(yaml_candidates) != 1:
    raise ValueError(f'Expected one data.yaml beside train/valid/test; found: {yaml_candidates}')

SOURCE_ROOT = yaml_candidates[0].parent
source_config = yaml.safe_load(yaml_candidates[0].read_text(encoding='utf-8'))
class_names = source_config.get('names')
if isinstance(class_names, dict):
    class_names = [class_names[index] for index in sorted(class_names)]
if class_names != ['Healthy', 'sigatoka']:
    raise ValueError(f'Unexpected class order/names: {class_names!r}')

FEEDBACK_ARCHIVE = Path('/content/drive/MyDrive/LeafCare/LeafCare-reviewed-feedback.zip')
if FEEDBACK_ARCHIVE.is_file():
    FEEDBACK_ROOT = Path('/content/leafcare_feedback')
    FEEDBACK_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(FEEDBACK_ARCHIVE) as archive:
        feedback_root = FEEDBACK_ROOT.resolve()
        for member in archive.infolist():
            target = (FEEDBACK_ROOT / member.filename).resolve()
            if target != feedback_root and feedback_root not in target.parents:
                raise ValueError(f'Unsafe path in feedback archive: {member.filename}')
        archive.extractall(FEEDBACK_ROOT)
    feedback_image_root = FEEDBACK_ROOT / 'train' / 'images'
    feedback_label_root = FEEDBACK_ROOT / 'train' / 'labels'
    if feedback_image_root.is_dir():
        merged_feedback = 0
        for feedback_image in feedback_image_root.iterdir():
            if not feedback_image.is_file() or feedback_image.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}:
                continue
            target_image = SOURCE_ROOT / 'train' / 'images' / feedback_image.name
            target_label = SOURCE_ROOT / 'train' / 'labels' / f'{feedback_image.stem}.txt'
            feedback_label = feedback_label_root / target_label.name
            if not feedback_label.is_file():
                raise FileNotFoundError(f'Missing feedback label: {feedback_label}')
            if target_image.exists():
                if target_image.read_bytes() != feedback_image.read_bytes():
                    raise FileExistsError(f'Feedback image conflicts with existing image: {target_image}')
                continue
            shutil.copy2(feedback_image, target_image)
            shutil.copy2(feedback_label, target_label)
            merged_feedback += 1
        print('Merged reviewed training examples:', merged_feedback)

INITIAL_CHECKPOINT = EXTRACT_ROOT / 'initial_weights' / 'leafcare_sigatoka_best.pt'
if not INITIAL_CHECKPOINT.is_file():
    raise FileNotFoundError(f'Initial checkpoint is missing from the archive: {INITIAL_CHECKPOINT}')

print('Dataset root:', SOURCE_ROOT)
print('Fine-tuning from:', INITIAL_CHECKPOINT)
print('Classes:', class_names)

## Normalize labels and paths

YOLO detection labels have five values per row (`class x_center y_center width height`). Segmentation rows contain a polygon. For this detection model, polygon rows are converted to their enclosing bounding boxes; existing box rows are preserved. The notebook reports per-class object totals for every split so class balance is visible before training.

In [ ]:
from collections import Counter

IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
reports = {}
CLEAN_ROOT.mkdir(parents=True, exist_ok=True)

for split in ('train', 'valid', 'test'):
    source_images = SOURCE_ROOT / split / 'images'
    source_labels = SOURCE_ROOT / split / 'labels'
    target_images = CLEAN_ROOT / split / 'images'
    target_labels = CLEAN_ROOT / split / 'labels'
    if not source_images.is_dir():
        raise FileNotFoundError(f'Missing image directory: {source_images}')
    target_images.mkdir(parents=True, exist_ok=True)
    target_labels.mkdir(parents=True, exist_ok=True)

    image_paths = sorted(p for p in source_images.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES)
    if not image_paths:
        raise ValueError(f'No images found in {source_images}')
    counts = Counter()
    converted_polygons = 0
    missing_label_files = 0

    for image_path in image_paths:
        shutil.copy2(image_path, target_images / image_path.name)
        label_path = source_labels / f'{image_path.stem}.txt'
        out_path = target_labels / f'{image_path.stem}.txt'
        if not label_path.is_file():
            missing_label_files += 1
            out_path.write_text('', encoding='utf-8')
            continue

        output_rows = []
        for line_number, line in enumerate(label_path.read_text(encoding='utf-8').splitlines(), 1):
            parts = line.split()
            if not parts:
                continue
            try:
                values = [float(value) for value in parts]
                class_value = values[0]
                class_id = int(class_value)
            except (ValueError, OverflowError) as error:
                raise ValueError(f'{label_path}:{line_number}: non-numeric label') from error
            if class_value != class_id or class_id < 0 or class_id >= len(class_names):
                raise ValueError(f'{label_path}:{line_number}: invalid class id {values[0]}')
            coordinates = values[1:]
            if not all(math.isfinite(value) and 0 <= value <= 1 for value in coordinates):
                raise ValueError(f'{label_path}:{line_number}: coordinates must be finite and normalized to [0, 1]')

            if len(values) == 5:
                x_center, y_center, width, height = coordinates
            elif len(coordinates) >= 6 and len(coordinates) % 2 == 0:
                xs = coordinates[0::2]
                ys = coordinates[1::2]
                x_min, x_max = min(xs), max(xs)
                y_min, y_max = min(ys), max(ys)
                width, height = x_max - x_min, y_max - y_min
                x_center, y_center = (x_min + x_max) / 2, (y_min + y_max) / 2
                converted_polygons += 1
            else:
                raise ValueError(f'{label_path}:{line_number}: expected a box or polygon row, got {len(values)} values')

            if width <= 0 or height <= 0:
                raise ValueError(f'{label_path}:{line_number}: box has zero/negative area')
            output_rows.append(f'{class_id} {x_center:.8f} {y_center:.8f} {width:.8f} {height:.8f}')
            counts[class_id] += 1

        out_path.write_text('\n'.join(output_rows) + ('\n' if output_rows else ''), encoding='utf-8')

    reports[split] = {'images': len(image_paths), 'objects': dict(counts), 'polygons_converted': converted_polygons, 'missing_label_files': missing_label_files}
    print(f'{split}: {reports[split]}')

yaml_path = CLEAN_ROOT / 'data.yaml'
yaml_path.write_text(yaml.safe_dump({
    'path': str(CLEAN_ROOT),
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'names': class_names,
    'nc': len(class_names),
}, sort_keys=False), encoding='utf-8')

healthy = reports['train']['objects'].get(0, 0)
sigatoka = reports['train']['objects'].get(1, 0)
if healthy and sigatoka / healthy > 4:
    print(f'WARNING: training object counts are imbalanced ({class_names[0]}={healthy}, {class_names[1]}={sigatoka}). Add diverse, correctly labeled Healthy examples and retrain for a more balanced detector.')

In [ ]:
import torch
from ultralytics import YOLO

HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU is available. YOLOv8m training on CPU may take a long time. Select a GPU runtime in Colab if available.')

RUNS_ROOT = Path('/content/leafcare_runs')
model = YOLO(str(INITIAL_CHECKPOINT))
train_result = model.train(
    data=str(yaml_path),
    imgsz=896,
    batch=-1 if HAS_GPU else 2,
    epochs=60,
    patience=15,
    device=0 if HAS_GPU else 'cpu',
    workers=2 if HAS_GPU else 0,
    seed=42,
    deterministic=True,
    amp=HAS_GPU,
    close_mosaic=10,
    project=str(RUNS_ROOT),
    name='sigatoka_yolov8m',
    exist_ok=True,
    plots=True,
)

BEST_CHECKPOINT = Path(train_result.save_dir) / 'weights' / 'best.pt'
if not BEST_CHECKPOINT.is_file():
    raise FileNotFoundError(f'Training did not produce best.pt at {BEST_CHECKPOINT}')
print('Best checkpoint:', BEST_CHECKPOINT)

## Evaluate on the held-out test set

Do not choose confidence thresholds using the test split. Use the training/validation results to tune a threshold, then report the final test metrics once. Compare results with real phone-captured leaves and inspect false positives and missed lesions; mAP and confidence are not clinical/agricultural certainty.

In [ ]:
best_model = YOLO(str(BEST_CHECKPOINT))
test_metrics = best_model.val(
    data=str(yaml_path),
    split='test',
    imgsz=896,
    batch=1,
    workers=0,
    device=0 if HAS_GPU else 'cpu',
    plots=True,
    project=str(RUNS_ROOT),
    name='sigatoka_test_evaluation',
    exist_ok=True,
)
print(f"Test precision: {test_metrics.box.mp:.3f}")
print(f"Test recall:    {test_metrics.box.mr:.3f}")
print(f"Test mAP50:     {test_metrics.box.map50:.3f}")
print(f"Test mAP50-95:  {test_metrics.box.map:.3f}")

In [ ]:
from google.colab import files

DRIVE_OUTPUT = Path('/content/drive/MyDrive/LeafCare/training_outputs')
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
shutil.copy2(BEST_CHECKPOINT, DRIVE_OUTPUT / 'leafcare_sigatoka_best.pt')

for name in ('results.csv', 'results.png', 'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png'):
    artifact = Path(train_result.save_dir) / name
    if artifact.is_file():
        shutil.copy2(artifact, DRIVE_OUTPUT / name)

print('Saved checkpoint and available training plots to:', DRIVE_OUTPUT)
files.download(str(BEST_CHECKPOINT))

After inspecting the independent test metrics and predictions, download `leafcare_sigatoka_best.pt` from Drive to the PC and back up the current checkpoint before replacing it. The API reads the model path configured by `LEAFCARE_MODEL_PATH` (default: `weights/new_combined_best.pt`); restart the API after changing checkpoints. Keep the held-out test set out of training and keep the original dataset attribution.